# 1. Cleanup old processes

In [14]:
!pkill -u "$USER" -f '^clingcon '
!ps -aux | grep clingcon

unknown   412893 66.6  0.0  10352  3540 pts/1    Ss+  10:47   0:00 /bin/bash -c ps -aux | grep clingcon
unknown   412895  0.0  0.0   9692  2296 pts/1    S+   10:47   0:00 grep clingcon


# 2. Run clingcon on instance with timeout

In [4]:
import time, subprocess

timeMax = "86400"
instanceName = "green_circuit.lp"
coreCount = "32"

outfile = f"i_{instanceName}-t_{timeMax}-cpu_{coreCount}-{time.strftime('%Y%m%d_%H%M%S')}.log"
print(outfile)

sessionName = "asp"

inner_cmd = f"""
    bash -lc '
        echo "================ INSTANCE BEGIN ================" > logs/{outfile};
        echo "File: instances/{instanceName}" >> logs/{outfile};
        echo "------------------------------------------------" >> logs/{outfile};
        cat instances/{instanceName} >> logs/{outfile};
        echo >> logs/{outfile};
        echo "================ CLINGCON OUTPUT ================" >> logs/{outfile};
        systemd-run --user --scope -p MemoryMax=128G -p MemoryHigh=120G -p MemorySwapMax=0 \
        clingcon --stats model2.lp game.lp instances/{instanceName} --opt-mode=ignore -n 1 --quiet=1 -t {coreCount} --time {timeMax} >> logs/{outfile} 2>&1
        '
"""

subprocess.run(["tmux", "new-session", "-d", "-s", sessionName, inner_cmd], check=True)
print(f"Started tmux session: {sessionName}")
print(f"Attach with: tmux attach -d -t {sessionName}")


i_green_circuit.lp-t_86400-cpu_32-20260318_105322.log
Started tmux session: asp
Attach with: tmux attach -d -t asp


# 3. Visualize

In [5]:
from clingcon_plotly_viewer import ClingconAssignmentViewer

#viewer = ClingconAssignmentViewer.from_log("logs/i_red_circuit.lp-t_86400-cpu_64-20260310_184714.log")
viewer = ClingconAssignmentViewer.from_log("logs/" + outfile)
ui = viewer.interactive_explorer_widget(cols=4, show_zeros_3d=True)
display(ui)

FileNotFoundError: [Errno 2] No such file or directory: 'logs/i_red_circuit.lp-t_86400-cpu_64-20260310_184714.log'